In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [2]:
df = pd.read_csv("../artifacts/data/01-raw/untouched_raw_original.csv")

In [3]:
df.shape

(15022, 29)

In [4]:
df["locality"].nunique()

73

Due to the nature of the dataset; some guardrails were put in place to have a valid split. Also due to insufficient data points at the intial stage, we use just a train and val split

- Collapse by locality to adequately represent locality distribution
- A stratified sampling technique was adopted to prevent domination by some classes
- Finally, a price band analysis was done on the two sets to determine if price distribution was equally represented in the two sets


In [5]:
MIN_LISTINGS = 50

locality_counts = df["locality"].value_counts()
rare_localitys = locality_counts[locality_counts < MIN_LISTINGS].index

df["locality_grouped"] = df["locality"].where(
    ~df["locality"].isin(rare_localitys), other="OTHER"
)

In [6]:
# Spliting

train_df, eval_df = train_test_split(
    df, test_size=0.2, random_state=2025, stratify=df["locality_grouped"]
)

In [7]:
def audit_price_bands(df, name):
    print(f"\n{name} price band distribution")
    print(pd.qcut(df["price"], 4).value_counts(normalize=True).sort_index())


audit_price_bands(train_df, "TRAIN")
audit_price_bands(eval_df, "EVAL")



TRAIN price band distribution
price
(174.999, 2300.0]          0.251227
(2300.0, 5000.0]           0.280769
(5000.0, 13000.0]          0.221020
(13000.0, 1655500000.0]    0.246983
Name: proportion, dtype: float64

EVAL price band distribution
price
(199.999, 2300.0]       0.250582
(2300.0, 4600.0]        0.249917
(4600.0, 13000.0]       0.250915
(13000.0, 2171400.0]    0.248586
Name: proportion, dtype: float64


In [8]:
train_df.to_csv("../artifacts/data/01-raw/train_df.csv")
eval_df.to_csv("../artifacts/data/01-raw/eval_df.csv")